# Clustering 
1. Symptom text preprocessing  
2. Unsupervised clustering (phenotype discovery)  

In [ ]:
# Load and analyze the COVID case-level clean dataset
DATA_PATH_COVID_CLEAN = "/Users/ariellerothman/Desktop/Capstone/df_clean_imputed.csv"
df_clean = pd.read_csv(DATA_PATH_COVID_CLEAN, low_memory=False)
OUTDIR = "/Users/ariellerothman/Desktop/Capstone/Outputs"

print("Dataset loaded:")
print(f"Shape: {df_clean.shape}")
print(f"\nColumns: {df_clean.columns.tolist()}")
print(f"\nData types:\n{df_clean.dtypes}")

In [ ]:
# Load pre-imputed dataset (instead of re-running imputation)
imputed_data_path = '/Users/ariellerothman/Desktop/Capstone/Outputs/df_clean_imputed.csv'

if os.path.exists(imputed_data_path):
    df_clean = pd.read_csv(imputed_data_path, low_memory=False)
    print(f"✓ Loaded pre-imputed dataset from: {imputed_data_path}")
    print(f"  Shape: {df_clean.shape}")
else:
    print(f"⚠ Imputed dataset not found. Run imputation code first.")

In [ ]:
### Creating comorbiditiy

# ============================================================
# RULE-BASED COMORBIDITY INDICATORS (UPDATED + NEW CATEGORIES)
# - Adds endocrine/thyroid + musculoskeletal + mental health +
#   GI + neuro_headache (migraine) + lymphatic/edema (optional)
# - Treats "none/no/unknown/NKA/NKDA" etc. as missing text
# - Saves outputs to CSV by default (no pyarrow needed)
#   (Parquet is optional if pyarrow/fastparquet is installed)
# ============================================================

import os
import re
import numpy as np
import pandas as pd

OUTDIR = "/Users/ariellerothman/Desktop/Capstone/Outputs"
os.makedirs(OUTDIR, exist_ok=True)

# Update this to your dataframe name if needed:
# df_clean = <your cleaned case-level dataframe>

FREE_TEXT_FIELDS = ["HISTORY", "CUR_ILL", "ALLERGIES", "PRIOR_VAX", "LAB_DATA", "OTHER_MEDS"]

# Terms that should be treated as "no information" (so they become missing)
NULL_TOKENS = {
    "none", "no", "n", "na", "n/a", "unknown", "unk", "none known", "not known",
    "nka", "nkda", "no known allergies", "no known drug allergies",
    "denies", "denies allergies", "no allergies", "nil", "negative"
}

_keep = re.compile(r"[^a-z0-9\s\-\/]")
_ws = re.compile(r"\s+")

def normalize_text(x) -> str:
    """Normalize free-text: lowercase, remove urls/emails/specials, collapse whitespace,
    and convert null-like tokens to empty string."""
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    if not x:
        return ""

    # Common "null-like" entries
    if x in NULL_TOKENS:
        return ""

    # remove urls/emails
    x = re.sub(r"http\S+|www\S+|https\S+", " ", x)
    x = re.sub(r"\S+@\S+", " ", x)

    # keep alnum + spaces + hyphen + slash
    x = _keep.sub(" ", x)
    x = _ws.sub(" ", x).strip()

    # If after normalization it's basically a null token, drop it
    if x in NULL_TOKENS:
        return ""

    return x

# ---------- UPDATED CATEGORY MAP ----------
# Notes:
# - Add categories where your "other" bucket showed lots of valid clinical signal:
#   hypothyroidism/thyroid, arthritis/OA, migraine, depression/anxiety, GERD, edema/lymphedema.
# - Keep patterns conservative using word boundaries (\b) to reduce false positives.
CATEGORY_MAP = {
    "CUR_ILL": {
        "acute_infection": [
            r"\bcovid\b", r"\bsars[-\s]?cov[-\s]?2\b", r"\binfluenza\b", r"\bflu\b",
            r"\binfection\b", r"\bpneumonia\b", r"\bbronchitis\b", r"\bviral\b", r"\bbacterial\b"
        ],
        "chronic_cardiovascular": [
            r"\bhypertension\b", r"\bhtn\b", r"\bcad\b", r"\bcoronary\b", r"\bafib\b",
            r"\bheart failure\b", r"\bchf\b", r"\barrhythmia\b", r"\bmi\b", r"\bmyocardial infarction\b"
        ],
        "chronic_respiratory": [
            r"\basthma\b", r"\bcopd\b", r"\bemphysema\b", r"\bchronic obstructive\b",
            r"\bsleep apnea\b", r"\bosa\b"
        ],
        "metabolic_endocrine": [
            r"\bdiabetes\b", r"\bdm\b", r"\bdm2\b", r"\btype\s?2\b", r"\bobesity\b",
            r"\bhyperlipid\w*\b", r"\bdyslipid\w*\b"
        ],
        # NEW: thyroid/endocrine explicitly (often shows up in your OTHER examples)
        "endocrine_thyroid": [
            r"\bthyroid\b", r"\bhypothyroid\w*\b", r"\bhyperthyroid\w*\b",
            r"\bhashimoto\w*\b", r"\bgraves\b"
        ],
        "autoimmune_inflammatory": [
            r"\bautoimmune\b", r"\blupus\b", r"\bsle\b", r"\bra\b", r"\brheumatoid arthritis\b",
            r"\bmultiple sclerosis\b", r"\bms\b", r"\bibs\b", r"\bibd\b", r"\bceliac\b"
        ],
        "neurologic": [
            r"\bseizure\b", r"\bepilepsy\b", r"\bstroke\b", r"\btia\b", r"\bparkinson\w*\b",
            r"\bneuropath\w*\b", r"\bdementia\b", r"\balzheimer\w*\b"
        ],
        # NEW: migraine/headache bucket (showed up in your examples)
        "neuro_headache": [
            r"\bmigraine\w*\b", r"\bheadache\w*\b", r"\bcluster headache\b"
        ],
        "cancer_immunocompromised": [
            r"\bcancer\b", r"\bmalignan\w*\b", r"\bchemotherap\w*\b", r"\bradiation\b",
            r"\btransplant\b", r"\bhiv\b", r"\baids\b",
            r"\bimmunosupp\w*\b", r"\bimmunocompromis\w*\b"
        ],
        # NEW (optional): edema/lymphedema bucket (shows in your examples)
        "lymphatic_edema": [
            r"\blymph(edema|oedema)\b", r"\bedema\b", r"\bswelling\b"
        ],
    },

    "HISTORY": {
        "cardiovascular": [
            r"\bhtn\b", r"\bhypertension\b", r"\bcad\b", r"\bcoronary\b", r"\bafib\b",
            r"\bheart failure\b", r"\bchf\b", r"\bmi\b", r"\bmyocardial infarction\b",
            r"\bvalve\b", r"\baortic stenosis\b"
        ],
        "respiratory": [
            r"\basthma\b", r"\bcopd\b", r"\bemphysema\b", r"\bchronic obstructive\b",
            r"\bosa\b", r"\bsleep apnea\b"
        ],
        "metabolic": [
            r"\bdiabetes\b", r"\bdm\b", r"\bdm2\b", r"\bobesity\b",
            r"\bhyperlipid\w*\b", r"\bdyslipid\w*\b"
        ],
        # NEW: thyroid/endocrine (common in HISTORY OTHER)
        "endocrine_thyroid": [
            r"\bthyroid\b", r"\bhypothyroid\w*\b", r"\bhyperthyroid\w*\b",
            r"\bhashimoto\w*\b", r"\bgraves\b"
        ],
        # NEW: musculoskeletal (arthritis/OA showed in your OTHER samples)
        "musculoskeletal": [
            r"\barthritis\b", r"\boveruse\b", r"\bosteoarthritis\b", r"\boa\b",
            r"\bgout\b", r"\bfibromyalgia\b", r"\bchronic pain\b"
        ],
        # NEW: mental health (often appears in HISTORY strings)
        "mental_health": [
            r"\bdepress\w*\b", r"\banxiety\b", r"\bptsd\b", r"\bbipolar\b",
            r"\bschizo\w*\b"
        ],
        # NEW: GI (GERD appears a lot)
        "gastrointestinal": [
            r"\bgerd\b", r"\bacid reflux\b", r"\bgastroesophageal\b",
            r"\bibs\b", r"\bconstipat\w*\b", r"\bdiverticul\w*\b"
        ],
        "kidney": [
            r"\bckd\b", r"\bchronic kidney\b", r"\brenal\b", r"\bdialysis\b",
            r"\brenal failure\b", r"\bkidney transplant\b", r"\btransplant\b"
        ],
        "clotting": [
            r"\bdvt\b", r"\bpe\b", r"\bthromb\w*\b", r"\bclot\b",
            r"\bembol\w*\b", r"\bantiphospholipid\b"
        ],
        "autoimmune": [
            r"\bautoimmune\b", r"\blupus\b", r"\bsle\b", r"\bra\b",
            r"\bms\b", r"\bmultiple sclerosis\b"
        ],
        "cancer_immunocompromised": [
            r"\bcancer\b", r"\bmalignan\w*\b", r"\bchemotherap\w*\b", r"\bradiation\b",
            r"\btransplant\b", r"\bhiv\b", r"\baids\b", r"\bimmunosupp\w*\b"
        ],
        "neurologic": [
            r"\bstroke\b", r"\btia\b", r"\bseizure\b", r"\bepilepsy\b",
            r"\bmigraine\w*\b", r"\bneuropath\w*\b", r"\bparkinson\w*\b"
        ],
        # NEW (optional): lymphatic/edema
        "lymphatic_edema": [
            r"\blymph(edema|oedema)\b", r"\bedema\b"
        ],
    },

    "ALLERGIES": {
        "drug_allergy": [
            r"\bpenicillin\b", r"\bsulfa\b", r"\bnsaid\b", r"\baspirin\b",
            r"\bcephalosporin\b", r"\bceclor\b", r"\bkeflex\b", r"\bdoxycycline\b"
        ],
        "food_allergy": [
            r"\bpeanut\b", r"\bnut\b", r"\begg\b", r"\bmilk\b", r"\bshellfish\b", r"\bgarlic\b"
        ],
        "latex_other_contact": [
            r"\blatex\b", r"\bdetergent\b", r"\bfragrance\b"
        ],
        "anaphylaxis_history": [
            r"\banaphylax\w*\b", r"\bepi[-\s]?pen\b", r"\bepinephrine\b"
        ],
    },

    "PRIOR_VAX": {
        "prior_covid_vax": [r"\bcovid\b", r"\bpfizer\b", r"\bmoderna\b", r"\bjanssen\b", r"\bnovavax\b"],
        "prior_flu_vax": [r"\bflu\b", r"\binfluenza\b"],
        # broaden common vaccines
        "prior_other_vax": [
            r"\bshingrix\b", r"\bshingles\b", r"\btetanus\b", r"\brabies\b",
            r"\bh1n1\b", r"\bmmr\b", r"\bhepatitis\b", r"\bvaricella\b"
        ],
        "prior_vax_reaction": [r"\breaction\b", r"\ballergic\b", r"\banaphylax\w*\b", r"\bside effect\b", r"\bsyncope\b"],
    },

    "LAB_DATA": {
        "cardiac_markers": [r"\btroponin\b", r"\bbnp\b", r"\bck[-\s]?mb\b"],
        "coagulation": [r"\bd[-\s]?dimer\b", r"\binr\b", r"\bptt\b", r"\bfibrinogen\b"],
        "inflammation": [r"\bcrp\b", r"\besr\b", r"\bferritin\b"],
        "cbc": [r"\bwbc\b", r"\bplatelet\w*\b", r"\bhemoglobin\b", r"\bhgb\b"],
        # optional: vitals/procedures bucket if you want to reduce LAB_DATA__other size
        "vitals_procedures": [r"\bbp\b", r"\bhr\b", r"\bspo2\b", r"\bultrasound\b", r"\bct\b", r"\bmri\b", r"\bx[-\s]?ray\b"],
    },

    "OTHER_MEDS": {
        "anticoagulant": [
            r"\bwarfarin\b", r"\bheparin\b", r"\beliquis\b", r"\bxarelto\b",
            r"\bapixaban\b", r"\brivaroxaban\b"
        ],
        "statin": [r"\bstatin\b", r"\batorvastatin\b", r"\bsimvastatin\b", r"\brosuvastatin\b"],
        "immunosuppressant": [r"\bprednisone\b", r"\bsteroid\b", r"\bmethotrexate\b", r"\btacrolimus\b", r"\bcyclosporine\b"],
        "diabetes_meds": [r"\binsulin\b", r"\bmetformin\b", r"\bsemaglutide\b", r"\bozempic\b", r"\bglp[-\s]?1\b"],
        # NEW: thyroid meds commonly present in meds lists
        "thyroid_meds": [r"\blevothyroxine\b", r"\bsynthroid\b", r"\bliothyronine\b"],
        # NEW: antidepressants/anxiolytics common in VAERS meds lists
        "psych_meds": [r"\bsertraline\b", r"\bfluoxetine\b", r"\bcitalopram\b", r"\bescitalopram\b", r"\bbuspirone\b", r"\balprazolam\b"],
    },
}

def compile_map(category_map):
    compiled = {}
    for col, cats in category_map.items():
        compiled[col] = {cat: [re.compile(p) for p in pats] for cat, pats in cats.items()}
    return compiled

COMPILED_MAP = compile_map(CATEGORY_MAP)

def flag_any(text: str, patterns) -> int:
    if not text:
        return 0
    return int(any(p.search(text) for p in patterns))

# ----------------------------
# APPLY TO df_clean
# ----------------------------
df_clean = df_clean.copy()

for col in FREE_TEXT_FIELDS:
    if col not in df_clean.columns:
        continue

    norm_col = f"{col}_NORM"
    df_clean[norm_col] = df_clean[col].apply(normalize_text)

    # missingness indicator after null-token normalization
    df_clean[f"{col}_MISSING"] = (df_clean[norm_col].str.len() == 0).astype("int8")

    # dictionary flags
    if col in COMPILED_MAP:
        cat_cols = []
        for cat, pats in COMPILED_MAP[col].items():
            outcol = f"{col}__{cat}"
            df_clean[outcol] = df_clean[norm_col].apply(lambda t: flag_any(t, pats)).astype("int8")
            cat_cols.append(outcol)

        # "other" = has text but did not match any category
        df_clean[f"{col}__other"] = (
            (df_clean[f"{col}_MISSING"] == 0) & (df_clean[cat_cols].sum(axis=1) == 0)
        ).astype("int8")

# ----------------------------
# SAVE ENGINEERED FEATURES
# ----------------------------
engineered_cols = [c for c in df_clean.columns if "__" in c or c.endswith("_MISSING")]

comorb_df = df_clean[["VAERS_ID"] + engineered_cols].copy()

# 1) Always save CSV (no extra deps)
csv_path = os.path.join(OUTDIR, "comorbidity_indicators.csv")
comorb_df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path, "shape:", comorb_df.shape)

# 2) Optional Parquet (only if engine installed)
parquet_path = os.path.join(OUTDIR, "comorbidity_indicators.parquet")
try:
    comorb_df.to_parquet(parquet_path, index=False)
    print("Saved Parquet:", parquet_path)
except Exception as e:
    print("Parquet not saved (missing engine). CSV is saved and sufficient.")
    print("Parquet error:", repr(e))

# ----------------------------
# QUICK DIAGNOSTICS
# ----------------------------
print("\nIndicator count:", len(engineered_cols))
top_counts = comorb_df[engineered_cols].sum().sort_values(ascending=False).head(30)
diag_df = pd.DataFrame({"count": top_counts, "percent": (top_counts / len(comorb_df) * 100).round(3)})
print("\nTop indicators by prevalence:")
print(diag_df)

# Optional: show "OTHER" examples for each field to see what you're missing
def show_other_examples(field, n=10):
    other_col = f"{field}__other"
    if other_col not in df_clean.columns:
        return
    mask = (df_clean[other_col] == 1) & (df_clean[field].notna())
    print(f"\n{field} OTHER examples (n={n}):")
    print(df_clean.loc[mask, field].head(n))

for fld in ["HISTORY", "CUR_ILL", "ALLERGIES", "PRIOR_VAX", "LAB_DATA", "OTHER_MEDS"]:
    if fld in df_clean.columns:
        show_other_examples(fld, n=10)

In [ ]:
# ============================================================
# 17. SYMPTOM TEXT PREPROCESSING
#     Updated supervised leakage cleanup with additional
#     quasi-administrative terms based on current model output
# ============================================================

import re
import pandas as pd

# --------------------------------------------------
# Base text cleaning
# --------------------------------------------------
_url_re = re.compile(r"http\S+|www\S+|https\S+")
_email_re = re.compile(r"\S+@\S+")
_num_re = re.compile(r"\d+")
_keep_re = re.compile(r"[^a-z0-9\s\-]")
_ws2 = re.compile(r"\s+")

def clean_symptom_text(x):
    if pd.isna(x) or x == "":
        return ""
    x = str(x).strip().lower()
    x = _url_re.sub(" ", x)
    x = _email_re.sub(" ", x)
    x = _num_re.sub(" __NUM__ ", x)
    x = _keep_re.sub(" ", x)
    x = _ws2.sub(" ", x).strip()
    return x

df_clean["SYMPTOM_TEXT_CLEAN"] = df_clean["SYMPTOM_TEXT"].apply(clean_symptom_text)

# --------------------------------------------------
# Strong supervised leakage scrubbing for severity prediction
# Removes direct outcome / adjudication / downstream-care terms
# plus residual administrative/report-style phrases
# --------------------------------------------------
LEAKAGE_PATTERNS = [re.compile(p) for p in [

    # ----------------------------------------------
    # Hospital / admission / downstream care
    # ----------------------------------------------
    r"\bhospital\w*\b",
    r"\badmit\w*\b",
    r"\binpatient\b",
    r"\bdischarg\w*\b",
    r"\bed\b",
    r"\ber\b",
    r"\bemergency\b",
    r"\bemergency room\b",
    r"\burgent\b",
    r"\burgent care\b",
    r"\btransport\w*\b",
    r"\btransfer\w*\b",
    r"\bambulance\b",
    r"\bems\b",
    r"\bhospice\b",

    # ----------------------------------------------
    # Death / life-threatening / rescue care
    # ----------------------------------------------
    r"\bdeath\b",
    r"\bdied\b",
    r"\bdead\b",
    r"\bfatal\b",
    r"\bpassed away\b",
    r"\bautopsy\b",
    r"\blife[-\s]?threaten\w*\b",
    r"\bintubat\w*\b",
    r"\bventilat\w*\b",
    r"\bicu\b",
    r"\barrest\b",

    # ----------------------------------------------
    # Seriousness / adjudication labels
    # ----------------------------------------------
    r"\bserious\w*\b",
    r"\bnon[-\s]?serious\w*\b",
    r"\bmedically significant\b",
    r"\bmedically important\b",
    r"\bcriterion\b",
    r"\bcriteria\b",
    r"\boutcome\b",
    r"\bdisability\b",
    r"\bpermanent\b",
    r"\bprolonged\b",

    # ----------------------------------------------
    # Administrative / report-template phrasing
    # ----------------------------------------------
    r"\breported cause\b",
    r"\breport medically\b",
    r"\bcriterion hospitalization\b",
    r"\bhospitalization medically\b",
    r"\bsignificant outcome\b",
    r"\bnarrative\b",
    r"\breport\b",
    r"\breported\b",
    r"\bspontaneous\b",
    r"\bcase\b",
    r"\bsource\b",
    r"\bpatient unknown\b",
    r"\bunknown\b",

    # ----------------------------------------------
    # Residual report-style / quasi-administrative phrases
    # seen in current model outputs
    # ----------------------------------------------
    r"\bevent resulted\b",
    r"\bevents resulted\b",
    r"\bresulted\b",
    r"\bcaused prolonged\b",
    r"\bmedical center\b",
    r"\bclinic care\b",
    r"\bwent care\b",
    r"\bvisited\b",
    r"\bperformed\b",

    # ----------------------------------------------
    # Other report-level phrasing seen previously
    # ----------------------------------------------
    r"\broom\b",
    r"\bdepartment\b",
    r"\bvisit\b",
    r"\bwent room\b",
    r"\bwent doctor\b",
    r"\bdoctor visit\b"
]]

def scrub_leakage_terms_strong(x):
    if pd.isna(x) or x == "":
        return ""
    x = str(x)
    for p in LEAKAGE_PATTERNS:
        x = p.sub(" ", x)
    x = _ws2.sub(" ", x).strip()
    return x

df_clean["SYMPTOM_TEXT_CLEAN_NOLEAK"] = (
    df_clean["SYMPTOM_TEXT_CLEAN"]
    .apply(scrub_leakage_terms_strong)
)

# --------------------------------------------------
# Clustering-specific administrative cleanup
# Keep separate from supervised no-leak field
# --------------------------------------------------
admin_phrases = [
    # vaccine / manufacturer boilerplate
    r"\bcovid vaccine\b",
    r"\bcovid immunisation\b",
    r"\bmrna moderna\b",
    r"\bmoderna covid\b",
    r"\bpfizer[-\s]?biontech\b",
    r"\bbiontech\b",
    r"\bmoderna\b",
    r"\bpfizer\b",
    r"\bjanssen\b",
    r"\bnovavax\b",
    r"\bbnt\b",
    r"\bmrna\b",
    r"\bvaccine\b",
    r"\bcovid\b",

    # lot / batch / product fields
    r"\blot number\b",
    r"\bbatch number\b",
    r"\bbatch lot\b",
    r"\bvaccine lot\b",
    r"\blot\b",
    r"\bbatch\b",
    r"\bdosage form\b",
    r"\bform\b",
    r"\broute administration\b",
    r"\broute admin\b",
    r"\bunspecified route\b",
    r"\badministration\b",
    r"\bintramuscular\b",
    r"\bsuspension injection\b",
    r"\bsuspension\b",

    # generic case-report boilerplate
    r"\bpatient received\b",
    r"\bpatient experienced\b",
    r"\bsubject experienced\b",
    r"\bspontaneous report\b",
    r"\bmedical history\b",
    r"\bmedical information\b",
    r"\bfollowing information\b",
    r"\bdescribes occurrence\b",
    r"\boccurrence\b",
    r"\bprovided\b",
    r"\bcontactable\b",
    r"\bcase\b",
    r"\breporter\b",
    r"\breported\b",
    r"\breport\b",
    r"\boutcome\b",
    r"\bunknown date\b",
    r"\bdate patient\b",
    r"\bdate\b",

    # dosing / process boilerplate
    r"\bsingle dose\b",
    r"\bdose\b",
    r"\bbooster\b",
    r"\badmin\b",
    r"\bprophylactic vaccination\b",
    r"\breceived\b",
    r"\bgiven\b",
    r"\badministered\b",
    r"\buse\b",

    # recurring vague metadata terms
    r"\bunknown\b",
    r"\binformation\b",
    r"\bdescribed\b",
    r"\bdescribes\b",
    r"\boccurred\b",
    r"\bpatient\b",
    r"\bsubject\b"
]

admin_regexes = [re.compile(p) for p in admin_phrases]

def remove_admin_terms(text):
    if pd.isna(text) or text == "":
        return ""
    text = str(text).lower()
    for rgx in admin_regexes:
        text = rgx.sub(" ", text)
    text = _ws2.sub(" ", text).strip()
    return text

df_clean["SYMPTOM_TEXT_CLEAN_CLUSTER"] = (
    df_clean["SYMPTOM_TEXT_CLEAN"]
    .apply(remove_admin_terms)
)

# --------------------------------------------------
# Optional token count after cluster cleanup
# --------------------------------------------------
df_clean["CLUSTER_TOKEN_COUNT"] = (
    df_clean["SYMPTOM_TEXT_CLEAN_CLUSTER"]
    .str.split()
    .str.len()
    .fillna(0)
    .astype(int)
)

# Optional clustering-specific subset
df_cluster = df_clean[df_clean["CLUSTER_TOKEN_COUNT"] >= 3].copy()

print("Rows in full dataset:", len(df_clean))
print("Rows retained for clustering:", len(df_cluster))
print("Rows dropped as near-empty after cluster cleanup:", len(df_clean) - len(df_cluster))

# --------------------------------------------------
# Save cleaned text variants
# --------------------------------------------------
df_clean[
    [
        "VAERS_ID",
        "SYMPTOM_TEXT",
        "SYMPTOM_TEXT_CLEAN",
        "SYMPTOM_TEXT_CLEAN_NOLEAK",
        "SYMPTOM_TEXT_CLEAN_CLUSTER",
        "CLUSTER_TOKEN_COUNT"
    ]
].to_csv(
    f"{OUTDIR}/symptom_text_clean_variants.csv",
    index=False
)

print([c for c in df_clean.columns if "SYMPTOM" in c or "CLUSTER_TOKEN_COUNT" in c])


In [ ]:
# ============================================================
# 18. CLUSTERING PREP: TF-IDF + SVD ON CLUSTER-CLEANED TEXT
# ============================================================

tfidf_cluster = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.9,
    stop_words="english",
    sublinear_tf=True,
    norm="l2",
    strip_accents="unicode",
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]+\b|__NUM__"
)

X_tfidf_cluster = tfidf_cluster.fit_transform(df_clean["SYMPTOM_TEXT_CLEAN_CLUSTER"])
print("TF-IDF shape:", X_tfidf_cluster.shape)

# Save TF-IDF + vocab
sp.save_npz(f"{OUTDIR}/symptom_text_tfidf_cluster_cleaned.npz", X_tfidf_cluster)
pd.DataFrame({"term": tfidf_cluster.get_feature_names_out()}).to_csv(
    f"{OUTDIR}/tfidf_vocabulary_cluster_cleaned.csv", index=False
)

# SVD / LSA
svd_cluster = TruncatedSVD(n_components=200, random_state=42)
X_lsa_cluster = svd_cluster.fit_transform(X_tfidf_cluster)
X_lsa_cluster = Normalizer(copy=False).fit_transform(X_lsa_cluster)

np.save(f"{OUTDIR}/symptom_text_lsa_200d_cluster_cleaned.npy", X_lsa_cluster)
np.save(f"{OUTDIR}/svd_components_200d_cluster_cleaned.npy", svd_cluster.components_)

print("X_lsa_cluster shape:", X_lsa_cluster.shape)

In [ ]:
# ============================================================
# 19. MODEL SELECTION: ELBOW + SAMPLED SILHOUETTE
# ============================================================

rng = np.random.default_rng(42)
silhouette_sample_size = 20000
silhouette_idx = rng.choice(X_lsa_cluster.shape[0], size=silhouette_sample_size, replace=False)
X_sil = X_lsa_cluster[silhouette_idx]

k_values = [5, 8, 10, 12, 15]
elbow_results = []

for k in k_values:
    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=42,
        batch_size=10000,
        n_init=3
    )
    labels_full = km.fit_predict(X_lsa_cluster)
    labels_sil = labels_full[silhouette_idx]
    sil = silhouette_score(X_sil, labels_sil)

    elbow_results.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": sil
    })

    print(f"k={k:2d} | silhouette={sil:.4f} | inertia={km.inertia_:.2e}")

elbow_df = pd.DataFrame(elbow_results)
elbow_df.to_csv(f"{OUTDIR}/cluster_selection_metrics_cluster_cleaned.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(elbow_df["k"], elbow_df["inertia"], marker="o")
ax[0].set_title("Elbow Method")
ax[0].set_xlabel("k")
ax[0].set_ylabel("Inertia")
ax[1].plot(elbow_df["k"], elbow_df["silhouette"], marker="o")
ax[1].set_title("Sampled Silhouette")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Silhouette score")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 20. STABILITY ANALYSIS (ARI ACROSS RANDOM SEEDS)
# ============================================================

# Set chosen k here after reviewing elbow + silhouette
best_k = 8

stability_sample_size = 50000
stability_idx = rng.choice(X_lsa_cluster.shape[0], size=stability_sample_size, replace=False)
X_stability = X_lsa_cluster[stability_idx]

seeds = [1, 7, 21, 42, 88]
labels_by_seed = {}

for seed in seeds:
    km = MiniBatchKMeans(
        n_clusters=best_k,
        random_state=seed,
        batch_size=10000,
        n_init=3
    )
    labels_by_seed[seed] = km.fit_predict(X_stability)

stability_results = []
for i, seed_a in enumerate(seeds):
    for seed_b in seeds[i+1:]:
        ari = adjusted_rand_score(labels_by_seed[seed_a], labels_by_seed[seed_b])
        stability_results.append({"seed_a": seed_a, "seed_b": seed_b, "ARI": ari})

stability_df = pd.DataFrame(stability_results)
print(stability_df)
print("\nMean ARI:", stability_df["ARI"].mean())

stability_df.to_csv(f"{OUTDIR}/cluster_stability_ari_cluster_cleaned.csv", index=False)

In [ ]:
# ============================================================
# 21. FINAL CLUSTERING MODEL ON FULL DATASET
# ============================================================

final_km = MiniBatchKMeans(
    n_clusters=best_k,
    random_state=42,
    batch_size=10000,
    n_init=5
)

df_clean["CLUSTER"] = final_km.fit_predict(X_lsa_cluster)

cluster_sizes = df_clean["CLUSTER"].value_counts().sort_index()
print(cluster_sizes)
(cluster_sizes / len(df_clean)).rename("proportion").to_csv(f"{OUTDIR}/cluster_sizes_cluster_cleaned.csv")

In [ ]:
# ============================================================
# 22. INTERPRET CLUSTERS USING RECONSTRUCTED TERM WEIGHTS
# ============================================================

feature_names = pd.read_csv(f"{OUTDIR}/tfidf_vocabulary_cluster_cleaned.csv")["term"].values
svd_components = np.load(f"{OUTDIR}/svd_components_200d_cluster_cleaned.npy")

reconstructed_centroids = final_km.cluster_centers_ @ svd_components

top_terms_rows = []
for cluster_id in range(best_k):
    centroid = reconstructed_centroids[cluster_id]
    top_idx = np.argsort(centroid)[::-1][:20]
    top_terms = feature_names[top_idx]

    print(f"\nCluster {cluster_id} top terms:")
    print(", ".join(top_terms))

    for rank, term in enumerate(top_terms, start=1):
        top_terms_rows.append({"cluster": cluster_id, "rank": rank, "term": term})

top_terms_df = pd.DataFrame(top_terms_rows)
top_terms_df.to_csv(f"{OUTDIR}/cluster_top_terms_cluster_cleaned.csv", index=False)

In [ ]:
# ============================================================
# 23. CLUSTER CHARACTERIZATION
# ============================================================

# Cluster vs serious outcome
cluster_serious = pd.crosstab(df_clean["CLUSTER"], df_clean["SERIOUS"], normalize="index")
print("\nCluster vs SERIOUS")
print(cluster_serious)
cluster_serious.to_csv(f"{OUTDIR}/cluster_vs_serious_cluster_cleaned.csv")

# Cluster vs sex
if "SEX" in df_clean.columns:
    cluster_sex = pd.crosstab(df_clean["CLUSTER"], df_clean["SEX"], normalize="index")
    print("\nCluster vs SEX")
    print(cluster_sex)
    cluster_sex.to_csv(f"{OUTDIR}/cluster_vs_sex_cluster_cleaned.csv")

# Cluster vs age
if "AGE_YRS" in df_clean.columns:
    cluster_age = df_clean.groupby("CLUSTER")["AGE_YRS"].describe()
    print("\nCluster age summary")
    print(cluster_age)
    cluster_age.to_csv(f"{OUTDIR}/cluster_age_summary_cluster_cleaned.csv")

# Cluster vs manufacturer
manufacturer_cols = [c for c in df_clean.columns if c.startswith("MANU__")]
if manufacturer_cols:
    cluster_manu = df_clean.groupby("CLUSTER")[manufacturer_cols].mean()
    print("\nCluster vs manufacturer")
    print(cluster_manu)
    cluster_manu.to_csv(f"{OUTDIR}/cluster_vs_manufacturer_cluster_cleaned.csv")

# Cluster vs comorbidities
comorb_cols = [c for c in df_clean.columns if "__" in c and not c.startswith("MANU__")]
if comorb_cols:
    cluster_comorb = df_clean.groupby("CLUSTER")[comorb_cols].mean()
    print("\nCluster vs comorbidities (first 10 cols)")
    print(cluster_comorb.iloc[:, :10])
    cluster_comorb.to_csv(f"{OUTDIR}/cluster_vs_comorbidities_cluster_cleaned.csv")

# Cluster summary
cluster_summary = df_clean.groupby("CLUSTER").agg({
    "SERIOUS": "mean",
    "AGE_YRS": "mean",
    "VAERS_ID": "count"
}).rename(columns={"SERIOUS": "SERIOUS_RATE", "AGE_YRS": "MEAN_AGE", "VAERS_ID": "N"})

print("\nCluster summary")
print(cluster_summary)
cluster_summary.to_csv(f"{OUTDIR}/cluster_summary_cluster_cleaned.csv")